# Exceptions and File Input/Output

**Seong Cho** · ACME Python Essentials, Lab 6

**Lab objective:** learn exception syntax and file interaction protocols — exceptions regulate program usage and report problems, and are essential for safely reading data from and writing data to external files.

<sub>Lab specification from the *Python Essentials* lab manual of [Foundations of Applied Mathematics](https://foundations-of-applied-mathematics.github.io/) (BYU ACME), © Dr. J. Humpherys, used under [CC BY 3.0 US](http://creativecommons.org/licenses/by/3.0/us/). Problem statements are quoted from the spec; the notes are condensed from it. Solutions are my own.</sub>

## Exceptions

An *exception* formally indicates an error and terminates the program early. Some of the more common exception types:

| Exception | Indication |
|---|---|
| `AttributeError` | An attribute reference or assignment failed. |
| `ImportError` | An `import` statement failed. |
| `IndexError` | A sequence subscript was out of range. |
| `NameError` | A local or global name was not found. |
| `TypeError` | An operation or function was applied to an object of inappropriate type. |
| `ValueError` | An operation or function received an argument that had the right type but an inappropriate value. |
| `ZeroDivisionError` | The second argument of a division or modulo operation was zero. |

In this notebook, examples that would halt with a traceback are wrapped in `try`/`except` (explained below) so the error message is printed instead.

In [1]:
for snippet in ["print(x)", "[1, 2, 3].fly()"]:
    try:
        eval(snippet)
    except Exception as e:
        print(type(e).__name__ + ":", e)

NameError: name 'x' is not defined
AttributeError: 'list' object has no attribute 'fly'


### Raising exceptions

Exceptions can also be used intentionally to signal a problem to the user or programmer: use the keyword `raise` followed by an exception class (optionally with an error message). As soon as an exception is raised, the program stops running unless the exception is handled.

```python
>>> if 7 is not 7.0:            # Raise an exception with an error message.
...     raise Exception("ints and floats are different!")
...
Traceback (most recent call last):
  File "<stdin>", line 2, in <module>
Exception: ints and floats are different!
```

In [2]:
try:
    for x in range(10):
        if x > 5:                       # Raise a specific kind of exception.
            raise ValueError("'x' should not exceed 5.")
        print(x, end=' ')
except ValueError as e:
    print("\nValueError:", e)

0 1 2 3 4 5 
ValueError: 'x' should not exceed 5.


## Problem 1: Validating `arithmagic()`

> **Problem 1.** Consider the following arithmetic "magic" trick.
> 1. Choose a 3-digit number where the first and last digits differ by 2 or more (say, 123).
> 2. Reverse this number by reading it backwards (321).
> 3. Calculate the positive difference of these numbers (321 − 123 = 198).
> 4. Add the reverse of the result to itself (198 + 891 = 1089).
>
> The result of the last step will always be 1089, regardless of the original number chosen in step 1 (can you explain why?).
>
> The following function prompts the user for input at each step of the magic trick, but does not check that the user's inputs are correct.
>
> ```python
> def arithmagic():
>     step_1 = input("Enter a 3-digit number where the first and last "
>                                            "digits differ by 2 or more: ")
>     step_2 = input("Enter the reverse of the first number, obtained "
>                                               "by reading it backwards: ")
>     step_3 = input("Enter the positive difference of these numbers: ")
>     step_4 = input("Enter the reverse of the previous result: ")
>     print(str(step_3), "+", str(step_4), "= 1089 (ta-da!)")
> ```
>
> Modify `arithmagic()` so that it verifies the user's input at each step. Raise a `ValueError` with an informative error message if any of the following occur:
> - The first number (`step_1`) is not a 3-digit number.
> - The first number's first and last digits differ by less than 2.
> - The second number (`step_2`) is not the reverse of the first number.
> - The third number (`step_3`) is not the positive difference of the first two numbers.
> - The fourth number (`step_4`) is not the reverse of the third number.
>
> (Hint: `input()` always returns a string, so each variable is a string initially. Use `int()` to cast the variables as integers when necessary. The built-in function `abs()` may also be useful.)

In [3]:
def arithmagic():
    step_1 = input("Enter a 3-digit number where the first and last "
                                           "digits differ by 2 or more: ")
    if not step_1.isdigit():
        raise ValueError("The first number (step_1) is not a 3-digit number.")
    else:
        if len(step_1) != 3:
            raise ValueError("The first number (step_1) is not a 3-digit number.")

    if abs(int(step_1[0]) - int(step_1[2])) < 2:
        raise ValueError("The first number’s first and last digits differ by less than 2.")

    step_2 = input("Enter the reverse of the first number, obtained "
                                              "by reading it backwards: ")
    if step_2[::-1] != step_1:
        raise ValueError("The second number (step_2) is not the reverse of the first number.")

    step_3 = input("Enter the positive difference of these numbers: ")
    if str(abs(int(step_1) - int(step_2))) != step_3:
        raise ValueError("The third number (step_3) is not the positive difference of the first two numbers.")

    step_4 = input("Enter the reverse of the previous result: ")
    if step_4[::-1] != step_3:
        raise ValueError("The fourth number (step_4) is not the reverse of the third number.")


    print(str(step_3), "+", str(step_4), "= 1089 (ta-da!)")

**Demo.** `input()` is replaced by a scripted user that echoes each answer. A correct run, then one bad answer for each of the five checks:

In [4]:
from unittest import mock

def scripted(answers):
    """Stand-in for input(): print the prompt and the next scripted answer."""
    answers = iter(answers)
    def fake_input(prompt=""):
        answer = next(answers)
        print(prompt + answer)
        return answer
    return fake_input

with mock.patch("builtins.input", scripted(["123", "321", "198", "891"])):
    arithmagic()

for answers in [["12"], ["121"], ["123", "322"], ["123", "321", "200"], ["123", "321", "198", "819"]]:
    print()
    try:
        with mock.patch("builtins.input", scripted(answers)):
            arithmagic()
    except ValueError as e:
        print("ValueError:", e)

Enter a 3-digit number where the first and last digits differ by 2 or more: 123
Enter the reverse of the first number, obtained by reading it backwards: 321
Enter the positive difference of these numbers: 198
Enter the reverse of the previous result: 891
198 + 891 = 1089 (ta-da!)

Enter a 3-digit number where the first and last digits differ by 2 or more: 12
ValueError: The first number (step_1) is not a 3-digit number.

Enter a 3-digit number where the first and last digits differ by 2 or more: 121
ValueError: The first number’s first and last digits differ by less than 2.

Enter a 3-digit number where the first and last digits differ by 2 or more: 123
Enter the reverse of the first number, obtained by reading it backwards: 322
ValueError: The second number (step_2) is not the reverse of the first number.

Enter a 3-digit number where the first and last digits differ by 2 or more: 123
Enter the reverse of the first number, obtained by reading it backwards: 321
Enter the positive diffe

## Handling Exceptions

To keep an exception from halting the program, put the problematic lines in a `try` block, followed by an `except` block saying what to do if an exception occurs. In `except Exception as e`, the name `e` is the exception; printing it shows its message, and it can be re-raised with `raise e` or just `raise`.

The control flow can be extended like a sequence of `if`-`elif`-`else` blocks:
1. The `try` block is executed until an exception is raised (if at all).
2. An `except` statement specifying the same kind of exception "catches" it, and its block runs. There may be several `except` blocks, and one `except` may list several exception types in parentheses.
3. The `else` block is executed if an exception was *not* raised in the `try` block.
4. The `finally` block is always executed if it is included — even if a `return` statement or an uncaught exception occurs in any block after the `try`.

See https://docs.python.org/3/tutorial/errors.html for more examples.

In [5]:
try:
    print("Entering try block...")
    raise Exception("for no reason")
    print("No problem!")                # This line gets skipped.
except Exception as e:                  # Runs just after the exception is raised.
    print("There was a problem:", e)

for fire in [True, False]:
    try:
        print("Entering try block...", end='')
        house_on_fire = False
        if fire:
            raise ValueError("The house is on fire!")
    except (ValueError, TypeError) as e:    # Check for multiple kinds of exceptions.
        print("caught an exception.")
        house_on_fire = True
    else:                                   # Skipped if there was an exception.
        print("no exceptions raised.")
    finally:
        print("The house is on fire:", house_on_fire)

def implode():
    try:                                # Try to return immediately...
        return
    finally:                            # ...but 'finally' goes before 'return'.
        print("Goodbye, world!")

implode()

Entering try block...
There was a problem: for no reason
Entering try block...caught an exception.
The house is on fire: True
Entering try block...no exceptions raised.
The house is on fire: False
Goodbye, world!


> **Achtung!** An `except` statement with no specified exception type catches *any* exception raised in the `try` block, which can mistakenly mask unexpected errors. Always be specific about the kinds of exceptions you expect to encounter.

In [6]:
def divider(x, y):
    try:
        return x / yy                   # The misspelled yy raises a NameError.
    except:                             # Catch ANY exception.
        print("y must not equal zero!")

divider(2, 3)                           # A misleading message.

def divider(x, y):
    try:
        return x / yy
    except ZeroDivisionError:           # Specify an exception type.
        print("y must not equal zero!")

try:
    divider(2, 3)
except NameError as e:
    print("NameError:", e)              # Now the mistake is obvious.

y must not equal zero!
NameError: name 'yy' is not defined


## Problem 2: Interrupting a random walk

> **Problem 2.** A *random walk* is a path created by a sequence of random steps. The following function simulates a random walk by repeatedly adding or subtracting 1 to a running total.
>
> ```python
> from random import choice
>
> def random_walk(max_iters=1e12):
>     walk = 0
>     directions = [1, -1]
>     for i in range(int(max_iters)):
>         walk += choice(directions)
>     return walk
> ```
>
> A `KeyboardInterrupt` is a special exception that can be triggered at any time by entering `ctrl+c` (on most systems) in the keyboard. Modify `random_walk()` so that if the user raises a `KeyboardInterrupt` by pressing `ctrl+c` while the program is running, the function catches the exception and prints "Process interrupted at iteration $i$". If no `KeyboardInterrupt` is raised, print "Process completed". In both cases, return `walk` as before.

In [7]:
from random import choice


def random_walk(max_iters=1e12):
    iter = 0

    try:
        walk = 0
        directions = [1, -1]
        for i in range(int(max_iters)):
            walk += choice(directions)
            iter += 1

        print("Process completed")

    except KeyboardInterrupt as e:
        print("Process interrupted at iteration", iter)
    finally:
        return walk

**Demo.** With the default `max_iters=1e12` the walk runs until `ctrl+c`, so here it runs with a small `max_iters` and finishes. To show the interrupt path without a keyboard, `choice` is temporarily swapped for a version that raises `KeyboardInterrupt` on its 50,001st call, as if `ctrl+c` were pressed then.

In [8]:
import random
random.seed(6)
print("walk =", random_walk(max_iters=1e5))

def interrupt_after(n, real_choice=choice):
    calls = 0
    def fake_choice(seq):
        nonlocal calls
        calls += 1
        if calls > n:
            raise KeyboardInterrupt      # What ctrl+c raises.
        return real_choice(seq)
    return fake_choice

real_choice, choice = choice, interrupt_after(50000)
try:
    print("walk =", random_walk())       # Default max_iters=1e12.
finally:
    choice = real_choice

Process completed
walk = -268
Process interrupted at iteration 50000
walk = -278


> **Note.** The built-in exceptions are organized into a class hierarchy. For example, `ValueError` inherits from the generic `Exception` class, so a `ValueError` *is* an `Exception`, but an `Exception` is not a `ValueError`. See https://docs.python.org/3/library/exceptions.html for the complete list and the hierarchy.

In [9]:
try:
    raise ValueError("caught!")
except Exception as e:                  # A ValueError is an Exception.
    print(e)

try:
    try:
        raise Exception("not caught!")
    except ValueError as e:             # An Exception is not a ValueError.
        print(e)
except Exception as e:
    print("The inner except missed it:", repr(e))

caught!
The inner except missed it: Exception('not caught!')


## File Input and Output

A *file object* is an interface to a file stream for reading from or writing to external files. The built-in `open()` creates one from a file name and an editing *mode*:

- `'r'`: **read.** Open an existing file for reading. The file must already exist, or `open()` raises a `FileNotFoundError`. This is the default mode.
- `'w'`: **write.** Create a new file or *overwrite an existing file* (careful!) and open it for writing.
- `'x'`: **write new.** Create a new file and open it for writing. If the file already exists, `open()` raises a `FileExistsError`. This is a safer form of `'w'` because it never overwrites existing files.
- `'a'`: **append.** Open a file for writing, appending new data to the end of the file if it already exists.

The examples read [`hello_world.txt`](hello_world.txt), a small input file next to this notebook.

In [10]:
myfile = open("hello_world.txt", 'r')   # Open a file for reading.
print(myfile.read())                    # Print the contents of the file.
myfile.close()                          # Close the file connection.

Hello,
World!



### The `with` statement

An `IOError` indicates that some input or output operation has failed. A `try`-`finally` block can make sure a file stream is closed safely. The `with` statement does the same more simply: `with open(<filename>, <mode>) as <alias>:` opens the file for the indented block and closes it automatically at the end, even if an exception occurs. This is the preferred method when a file only needs to be accessed briefly.

In [11]:
myfile = open("hello_world.txt", 'r')
try:
    contents = myfile.readlines()       # Read in the content by line.
finally:
    myfile.close()                      # Explicitly close the file.

with open("hello_world.txt", 'r') as myfile:    # Equivalent.
    contents = myfile.readlines()
print(contents, myfile.closed)

['Hello,\n', 'World!\n'] True


### Reading and writing

Open file objects have an implicit *cursor* marking where to read or write. After the whole file has been read once, either close and reopen it or reset the cursor with `seek(0)` before reading again.

| Attribute | Description |
|---|---|
| `closed` | `True` if the object is closed. |
| `mode` | The access mode used to open the file object. |
| `name` | The name of the file. |

| Method | Description |
|---|---|
| `close()` | Close the connection to the file. |
| `read()` | Read a given number of bytes; with no input, read the entire file. |
| `readline()` | Read a line of the file, including the newline character at the end. |
| `readlines()` | Call `readline()` repeatedly and return a list of the resulting lines. |
| `seek()` | Move the cursor to a new position. |
| `tell()` | Report the current position of the cursor. |
| `write()` | Write a single string to the file (spaces are *not* added). |
| `writelines()` | Write a list of strings to the file (newline characters are *not* added). |

Only strings can be written to files; cast other types with `str()` first, and be mindful of spaces and newlines to separate the data. (The example's `out.txt` is removed at the end of the cell.)

In [12]:
import os

with open("out.txt", 'w') as outfile:   # Open 'out.txt' for writing.
    for i in range(10):
        outfile.write(str(i**2) + ' ')  # Write some strings (and spaces).
print(outfile.closed)                   # The file is closed automatically.

with open("out.txt", 'r') as infile:
    print(repr(infile.read()), infile.tell())
    infile.seek(0)                      # Reset the cursor to read again.
    print(repr(infile.read(5)))
os.remove("out.txt")

True
'0 1 4 9 16 25 36 49 64 81 ' 26
'0 1 4'


## Problem 3: `ContentFilter` constructor

> **Problem 3.** Define a class called `ContentFilter`. Implement the constructor so that it accepts the name of a file to be read.
> 1. If the file name is invalid in any way, prompt the user for another filename using `input()`. Continue prompting the user until they provide a valid filename.
>
>    ```python
>    >>> cf1 = ContentFilter("hello_world.txt")     # File exists.
>    >>> cf2 = ContentFilter("not-a-file.txt")      # File doesn't exist.
>    Please enter a valid file name: still-not-a-file.txt
>    Please enter a valid file name: hello_world.txt
>    >>> cf3 = ContentFilter([1, 2, 3])             # Not even a string.
>    Please enter a valid file name: hello_world.txt
>    ```
>
>    (Hint: `open()` might raise a `FileNotFoundError`, a `TypeError`, or an `OSError`.)
> 2. Read the file and store its name and contents as attributes (store the contents as a single string). Make sure the file is securely closed.

The full class, including the Problem 4 methods, is defined under Problem 4 after the string-formatting notes.

### String formatting

The `str` class has several methods for parsing and formatting strings, useful for processing data from a source file and preparing data to write to another.

| Method | Returns |
|---|---|
| `count()` | The number of times a given substring occurs within the string. |
| `find()` | The lowest index where a given substring is found. |
| `isalpha()` | `True` if all characters in the string are alphabetic (a, b, c, ...). |
| `isdigit()` | `True` if all characters in the string are digits (0, 1, 2, ...). |
| `isspace()` | `True` if all characters in the string are whitespace (`" "`, `'\t'`, `'\n'`). |
| `join()` | The concatenation of the strings in a given iterable with a specified separator between entries. |
| `lower()` | A copy of the string converted to lowercase. |
| `upper()` | A copy of the string converted to uppercase. |
| `replace()` | A copy of the string with occurrences of a given substring replaced by a different specified substring. |
| `split()` | A list of segments of the string, using a given character or string as a delimiter. |
| `strip()` | A copy of the string with leading and trailing whitespace removed. |

`join()` puts the principal string between the entries of a list of strings; `split()` does the reverse, splitting on a given delimiter, or on any whitespace if none is given.

In [13]:
words = ["state", "of", "the", "art"]
print("-".join(words))
print("One fish\nTwo fish\nRed fish\nBlue fish\n".split('\n'))
print("One fish\nTwo fish\nRed fish\nBlue fish\n".split())

state-of-the-art
['One fish', 'Two fish', 'Red fish', 'Blue fish', '']
['One', 'fish', 'Two', 'fish', 'Red', 'fish', 'Blue', 'fish']


Can you tell the difference between the following routines?

In [14]:
with open("hello_world.txt", 'r') as myfile:
    contents = myfile.readlines()
print(contents)

with open("hello_world.txt", 'r') as myfile:
    contents = myfile.read().split('\n')
print(contents)

['Hello,\n', 'World!\n']
['Hello,', 'World!', '']


## Problem 4: `ContentFilter` output methods

> **Problem 4.** Add the following methods to the `ContentFilter` class for writing the contents of the original file to new files. Each method should accept the name of a file to write to and a keyword argument `mode` that specifies the file access mode, defaulting to `'w'`. If `mode` is not `'w'`, `'x'`, or `'a'`, raise a `ValueError` with an informative message.
> 1. `uniform()`: write the data to the outfile with uniform case. Include an additional keyword argument `case` that defaults to `"upper"`.
>    If `case="upper"`, write the data in upper case. If `case="lower"`, write the data in lower case. If `case` is not one of these two values, raise a `ValueError`.
> 2. `reverse()`: write the data to the outfile in reverse order. Include an additional keyword argument `unit` that defaults to `"line"`.
>    If `unit="word"`, reverse the ordering of the words in each line, but write the lines in the same order as the original file. If `unit="line"`, reverse the ordering of the lines, but do not change the ordering of the words on each individual line. If `unit` is not one of these two values, raise a `ValueError`.
> 3. `transpose()`: write a "transposed" version of the data to the outfile. That is, write the first word of each line of the data to the first line of the new file, the second word of each line of the data to the second line of the new file, and so on. Viewed as a matrix of words, the rows of the input file then become the columns of the output file, and vice versa. You may assume that there are an equal number of words on each line of the input file.
>
> Also implement the `__str__()` magic method so that printing a `ContentFilter` object yields the following output. You may want to calculate these statistics in the constructor.
>
> ```text
> Source file:            <filename>
> Total characters:       <The total number of characters in the file>
> Alphabetic characters:  <The number of letters>
> Numerical characters:   <The number of digits>
> Whitespace characters:  <The number of spaces, tabs, and newlines>
> Number of lines:        <The number of lines>
> ```
>
> (Hint: list comprehensions are *very* useful for some of these functions. For example, what does `[line[::-1] for line in lines]` do? What about `sum([s.isspace() for s in data])`?)
>
> Compare your class to the following example.
>
> ```text
> # cf_example1.txt
> A b C
> d E f
> ```
>
> ```python
> >>> cf = ContentFilter("cf_example1.txt")
> >>> cf.uniform("uniform.txt", mode='w', case="upper")
> >>> cf.uniform("uniform.txt", mode='a', case="lower")
> >>> cf.reverse("reverse.txt", mode='w', unit="word")
> >>> cf.reverse("reverse.txt", mode='a', unit="line")
> >>> cf.transpose("transpose.txt", mode='w')
> ```
>
> ```text
> # uniform.txt       # reverse.txt       # transpose.txt
> A B C               C b A               A d
> D E F               f E d               b E
> a b c               d E f               C f
> d e f               A b C
> ```

In [15]:
# Problems 3 and 4: Write a 'ContentFilter' class.
class ContentFilter:

    def __init__(self, file_name):
        self.file_name = file_name
        self.contents = []
        self.ocontents = []
        self.open_file()
        self.clean_contents()
        self.char = 0
        self.alphachar = 0
        self.numchar = 0
        self.wschar = 0
        self.numlines = len(self.contents)
        self.count_characters()

    def open_file(self):
        try:
            with open(self.file_name, 'r') as myfile:
                self.ocontents = myfile.readlines()

        except (FileNotFoundError, TypeError, OSError) as e:
            self.file_name = input("Please enter a valid file name: ")
            self.open_file()

    def clean_contents(self):
        for i in range(len(self.ocontents)):
            nl = self.ocontents[i].find("\n")
            if nl != -1:
                self.contents.append(self.ocontents[i][:nl])
            else:
                self.contents.append(self.ocontents[i])

    def count_characters(self):
        for i in range(len(self.ocontents)):
            self.char += len(self.ocontents[i])
            for j in range(len(self.ocontents[i])):
                if self.ocontents[i][j].isalpha():
                    self.alphachar += 1
                elif self.ocontents[i][j].isdigit():
                    self.numchar += 1
                elif self.ocontents[i][j].isspace():
                    self.wschar += 1


    def uniform(self, outfile_name, mode='w', case="upper"):
        #handle exceptions for mode
        if mode != 'w' and mode != 'x' and mode != 'a':
            raise ValueError("Mode must be 'w', 'x', or 'a'!")
        #handle exceptions for case
        elif case != "upper" and case != "lower":
            raise ValueError("Case must be \"upper\" or \"lower\"!")
        #if the values are correct, run the method
        else:
            with open(outfile_name, mode) as outfile:
                if case == "upper":
                    for i in range(len(self.contents)):
                        outfile.write(self.contents[i].upper()+'\n')
                else:
                    for i in range(len(self.contents)):
                        outfile.write(self.contents[i].lower()+'\n')

    def reverse(self, outfile_name, mode='w', unit="line"):
        #handle exceptions for mode
        if mode != 'w' and mode != 'x' and mode != 'a':
            raise ValueError("Mode must be 'w', 'x', or 'a'!")
        #handle exceptions for unit
        elif unit != "line" and unit != "word":
            raise ValueError("Unit must be \"line\" or \"word\"!")
        else:
            with open(outfile_name, mode) as outfile:
                if unit == "line":
                    for i in range(len(self.contents)-1, -1, -1):
                        outfile.write(self.contents[i]+'\n')
                else:
                    for i in range(len(self.contents)):
                        # Fix: was self.contents[i][::-1], which reversed the characters
                        # ("bird" -> "drib"), not the order of the words.
                        outfile.write(" ".join(self.contents[i].split()[::-1])+'\n')

    def transpose(self, outfile_name, mode='w'):
        import numpy as np
        #handle exceptions for mode
        if mode != 'w' and mode != 'x' and mode != 'a':
            raise ValueError("Mode must be 'w', 'x', or 'a'!")
        elif len(self.contents) != 0:
            tr = []
            for i in range(len(self.contents)):
                tr.append(self.contents[i].split())

            tr = np.array(tr)
            tr = tr.T

            with open(outfile_name, mode) as outfile:
                for i in range(len(tr)):
                    for j in range(len(tr[i])):
                        if j != len(tr[i])-1:
                            outfile.write(tr[i][j]+' ')
                        else:
                            outfile.write(tr[i][j]+'\n')

    def __str__(self):
        s = "Source file:\t\t" + self.file_name + '\n'     # Fix: label was "SourceFile:".
        s += "Total characters:\t" + str(self.char) + '\n'
        s += "Alphabetic characters:\t" + str(self.alphachar) + '\n'
        s += "Numerical characters:\t" + str(self.numchar) + '\n'
        s += "Whitespace characters:\t" + str(self.wschar) + '\n'
        s += "Number of lines:\t" + str(self.numlines)
        return s

**Demo: Problem 3.** Invalid file names (a missing file, then a list) make the constructor prompt until a valid name is given; `input()` is scripted as before.

In [16]:
cf1 = ContentFilter("hello_world.txt")             # File exists.
with mock.patch("builtins.input", scripted(["still-not-a-file.txt", "hello_world.txt"])):
    cf2 = ContentFilter("not-a-file.txt")          # File doesn't exist.
with mock.patch("builtins.input", scripted(["hello_world.txt"])):
    cf3 = ContentFilter([1, 2, 3])                 # Not even a string.

print(cf1.file_name, cf2.file_name, cf3.file_name)
print(cf1.contents)
assert cf1.contents == cf2.contents == cf3.contents == ["Hello,", "World!"]

Please enter a valid file name: still-not-a-file.txt
Please enter a valid file name: hello_world.txt
Please enter a valid file name: hello_world.txt
hello_world.txt hello_world.txt hello_world.txt
['Hello,', 'World!']


**Demo: Problem 4.** The spec's example, run on [`cf_example1.txt`](cf_example1.txt) and checked against the expected files:

In [17]:
cf = ContentFilter("cf_example1.txt")
cf.uniform("uniform.txt", mode='w', case="upper")
cf.uniform("uniform.txt", mode='a', case="lower")
cf.reverse("reverse.txt", mode='w', unit="word")
cf.reverse("reverse.txt", mode='a', unit="line")
cf.transpose("transpose.txt", mode='w')

expected = {"uniform.txt":   "A B C\nD E F\na b c\nd e f\n",
            "reverse.txt":   "C b A\nf E d\nd E f\nA b C\n",
            "transpose.txt": "A d\nb E\nC f\n"}
for name, text in expected.items():
    with open(name) as f:
        written = f.read()
    print(f"# {name}\n{written}")
    assert written == text
print(cf)

# uniform.txt
A B C
D E F
a b c
d e f

# reverse.txt
C b A
f E d
d E f
A b C

# transpose.txt
A d
b E
C f

Source file:		cf_example1.txt
Total characters:	12
Alphabetic characters:	6
Numerical characters:	0
Whitespace characters:	6
Number of lines:	2


The example's one-letter words can't tell reversing the *words* from reversing the *characters*, so here is [`word.txt`](word.txt), along with `transpose()`:

In [18]:
cf = ContentFilter("word.txt")
cf.reverse("reverse.txt", unit="word")
cf.transpose("transpose.txt")
for name in ["word.txt", "reverse.txt", "transpose.txt"]:
    with open(name) as f:
        print(f"# {name}\n{f.read().rstrip()}\n")

with open("reverse.txt") as f:
    assert f.read() == "far plane bird\nchill rat person\n"

# word.txt
bird plane far
person rat chill

# reverse.txt
far plane bird
chill rat person

# transpose.txt
bird person
plane rat
far chill



The statistics in `__str__()` on [`hi.txt`](hi.txt), which mixes letters, digits, punctuation, and whitespace, checked against counts computed directly from the file:

In [19]:
cf = ContentFilter("hi.txt")
print(cf)

with open("hi.txt") as f:
    data = f.read()
assert cf.char == len(data)
assert cf.alphachar == sum([s.isalpha() for s in data])
assert cf.numchar == sum([s.isdigit() for s in data])
assert cf.wschar == sum([s.isspace() for s in data])
assert cf.numlines == len(data.splitlines())

Source file:		hi.txt
Total characters:	71
Alphabetic characters:	31
Numerical characters:	3
Whitespace characters:	35
Number of lines:	3


Invalid `mode`, `case`, or `unit` arguments raise a `ValueError`, and `mode='x'` refuses to overwrite. Finally, the output files are cleaned up.

In [20]:
for call in [lambda: cf.uniform("out.txt", mode='r'),
             lambda: cf.uniform("out.txt", case="title"),
             lambda: cf.reverse("out.txt", unit="letter"),
             lambda: cf.transpose("out.txt", mode='rw'),
             lambda: cf.transpose("transpose.txt", mode='x')]:
    try:
        call()
    except (ValueError, FileExistsError) as e:
        print(type(e).__name__ + ":", e)

for name in ["uniform.txt", "reverse.txt", "transpose.txt"]:
    os.remove(name)
assert not os.path.exists("out.txt")

ValueError: Mode must be 'w', 'x', or 'a'!
ValueError: Case must be "upper" or "lower"!
ValueError: Unit must be "line" or "word"!
ValueError: Mode must be 'w', 'x', or 'a'!
FileExistsError: [Errno 17] File exists: 'transpose.txt'


## Additional Material

### Custom exception classes

A custom exception is a class that inherits from an existing exception class, typically `Exception`. This may seem trivial, but the interpreter never raises a `TooHardError` on its own, so any `TooHardError` must have come from a hand-written `raise`, making the source of a problem easier to find.

### Chaining exceptions

Sometimes, especially in large programs, it is useful to raise one kind of exception just after catching another. `raise ... from e` links the two, so the traceback shows where the error originated (this feature was added in Python 3). The full traceback reads "The above exception was the direct cause of the following exception"; the link is stored in `__cause__`.

In [21]:
class TooHardError(Exception):
    pass

try:
    try:
        raise TooHardError("This lab is impossible!")
    except TooHardError as e:
        raise NotImplementedError("Lab is incomplete") from e
except NotImplementedError as e:
    print(repr(e), "<- caused by", repr(e.__cause__))

NotImplementedError('Lab is incomplete') <- caused by TooHardError('This lab is impossible!')


### More string formatting tools

`str.format()` inserts non-string values into a string: write the desired output with `{}` in place of each value, then pass the values in order. In the progress bar below, `"\r[{:<20}]"` returns the cursor to the start of the line, then prints the first argument left-aligned in at least 20 spaces; `end=''` keeps every print on the same line. Printing at every iteration dramatically slows the loop down; printing only once per `chunk` iterations, as here, avoids that. For a more robust progress bar, research the `tqdm` module. See https://docs.python.org/3/library/string.html#format-string-syntax.

In [22]:
day, month, year = 10, "June", 2017
print("Is today", day, str(month) + ',', str(year) + "?")
print("Is today {} {}, {}?".format(day, month, year))

iters = int(1e6)
chunk = iters // 20
for i in range(iters):
    if not i % chunk:
        print("\r[{:<20}] i = {}".format('='*((i//chunk)+1), i), end='', flush=True)
print()

Is today 10 June, 2017?
Is today 10 June, 2017?
[=                   ] i = 0

[==                  ] i = 50000

[===                 ] i = 100000

[====                ] i = 150000

[=====               ] i = 200000

[======              ] i = 250000

... (10 outputs left out)

[=================   ] i = 800000

[==================  ] i = 850000

[=================== ] i = 900000

[====================] i = 950000

### Standard library modules for I/O

| Module | Description |
|---|---|
| `csv` | CSV (comma separated value) file writing and parsing. |
| `io` | Support for file objects and `open()`. |
| `os` | Communication with the operating system. |
| `os.path` | Common path operations such as checking for file existence. |
| `pickle` | Create portable serialized representations of Python objects. |

See https://docs.python.org/3/library for details on each module.